In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# !pip install -q  langchain langchain-community langchain-core langchain-groq faiss-cpu sentence-transformers pandas
# !pip install -q --no-deps "langchain<0.3.0" "langchain-community<0.3.0" langchain-core langchain-groq faiss-cpu sentence-transformers pandas
# !pip install -q --no-deps langchain-huggingface langchain-community langchain-core langchain-groq faiss-cpu sentence-transformers pandas
!pip install -q --no-deps groq fastapi uvicorn pyngrok python-multipart nest-asyncio langchain-huggingface langchain-community langchain-core langchain-groq faiss-cpu sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.0 MB/s eta 0:00:00:00:0100:01


In [3]:
%%writefile config.py
"""
config.py

Central configuration for the EDA Synthesis Report Analyzer.
Updated to support Groq API models (Llama 3) for cloud execution.
"""

import os
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class Config:
    # --- Embedding Model (HuggingFace) ---
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Vector Store ---
    FAISS_INDEX_PATH: str = "./faiss_index"
    FAISS_INDEX_NAME: str = "eda_timing_index"

    # --- LLM Provider ---
    # Switch provider to "groq" for Kaggle/Cloud execution
    LLM_PROVIDER: str = "groq"             # "groq" | "ollama"
    
    # Groq Settings
    GROQ_MODEL: str = "llama-3.3-70b-versatile"
    GROQ_API_KEY: Optional[str] = os.getenv("GROQ_API_KEY")

    # Ollama Settings (Fallback)
    OLLAMA_MODEL: str = "mistral"
    OLLAMA_BASE_URL: str = "http://localhost:11434"

    # --- Retrieval ---
    TOP_K_PATHS: int = 5
    SELF_QUERY_VERBOSE: bool = True

    # --- UI ---
    APP_TITLE: str = "EDA Synthesis Report Analyzer"
    APP_ICON: str = "⚡"


CONFIG = Config()

Writing config.py


In [4]:
%%writefile rpt_parser.py
"""
rpt_parser.py

Parses synthesis timing report (.rpt) files and splits them into timing path blocks.
"""

import re
from dataclasses import dataclass, field
from typing import Optional

try:
    from langchain_core.documents import Document
except ImportError:
    @dataclass
    class Document:
        page_content: str
        metadata: dict = field(default_factory=dict)


PATH_BLOCK_PATTERN = re.compile(
    r"(Startpoint:.*?slack\s*\((VIOLATED|MET)\)\s*-?\d+\.\d+)",
    re.DOTALL,
)

STARTPOINT_PATTERN = re.compile(r"Startpoint:\s*(\S+)")
ENDPOINT_PATTERN = re.compile(r"Endpoint:\s*(\S+)")
PATH_GROUP_PATTERN = re.compile(r"Path Group:\s*(\S+)")
PATH_TYPE_PATTERN = re.compile(r"Path Type:\s*(min|max)", re.IGNORECASE)
SLACK_VALUE_PATTERN = re.compile(r"slack\s*\([A-Z]+\)\s*(-?\d+\.\d+)")

CELL_LINE_PATTERN = re.compile(
    r"^\s*(\S+)\s+\(([A-Za-z0-9_]+)\)\s+([\d.]+)\s+([\d.]+)\s+[rf]?\s*$",
    re.MULTILINE,
)


@dataclass
class ParsedPath:
    raw_text: str
    slack_status: str
    slack_value: Optional[float]
    startpoint: Optional[str]
    endpoint: Optional[str]
    path_group: Optional[str]
    path_type: Optional[str]
    module: Optional[str]
    top_cells: list


def _infer_module_name(startpoint: Optional[str], endpoint: Optional[str], block_text: str) -> Optional[str]:
    cell_match = CELL_LINE_PATTERN.search(block_text)
    if cell_match:
        instance_path = cell_match.group(1)
        if "/" in instance_path:
            return instance_path.split("/")[0]

    for identifier in (startpoint, endpoint):
        if identifier and "/" in identifier:
            return identifier.split("/")[0]
    return None


def _extract_top_cells(block_text: str, top_n: int = 3) -> list:
    rows = []
    for match in CELL_LINE_PATTERN.finditer(block_text):
        instance, cell_type, incr_delay, _cum_delay = match.groups()
        try:
            rows.append((instance, cell_type, float(incr_delay)))
        except ValueError:
            continue
    rows.sort(key=lambda r: r[2], reverse=True)
    return rows[:top_n]


def _parse_single_block(block_text: str, slack_status: str) -> ParsedPath:
    startpoint_match = STARTPOINT_PATTERN.search(block_text)
    endpoint_match = ENDPOINT_PATTERN.search(block_text)
    path_group_match = PATH_GROUP_PATTERN.search(block_text)
    path_type_match = PATH_TYPE_PATTERN.search(block_text)
    slack_value_match = SLACK_VALUE_PATTERN.search(block_text)

    startpoint = startpoint_match.group(1) if startpoint_match else None
    endpoint = endpoint_match.group(1) if endpoint_match else None
    path_group = path_group_match.group(1) if path_group_match else None
    path_type = path_type_match.group(1).lower() if path_type_match else None
    slack_value = float(slack_value_match.group(1)) if slack_value_match else None

    module = _infer_module_name(startpoint, endpoint, block_text)
    top_cells = _extract_top_cells(block_text)

    return ParsedPath(
        raw_text=block_text.strip(),
        slack_status=slack_status,
        slack_value=slack_value,
        startpoint=startpoint,
        endpoint=endpoint,
        path_group=path_group,
        path_type=path_type,
        module=module,
        top_cells=top_cells,
    )


def parse_rpt_text(report_text: str) -> list:
    parsed_paths = []
    for match in PATH_BLOCK_PATTERN.finditer(report_text):
        block_text, slack_status = match.group(1), match.group(2)
        parsed_paths.append(_parse_single_block(block_text, slack_status))
    return parsed_paths


def to_documents(parsed_paths: list) -> list:
    documents = []
    for path in parsed_paths:
        metadata = {
            "slack_status": path.slack_status,
            "slack_value": path.slack_value,
            "startpoint": path.startpoint,
            "endpoint": path.endpoint,
            "path_group": path.path_group,
            "path_type": path.path_type,
            "module": path.module,
            "top_cell_1": path.top_cells[0][0] if len(path.top_cells) > 0 else None,
            "top_cell_1_delay": path.top_cells[0][2] if len(path.top_cells) > 0 else None,
        }
        documents.append(Document(page_content=path.raw_text, metadata=metadata))
    return documents


def parse_rpt_file(file_path: str) -> list:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        report_text = f.read()

    parsed_paths = parse_rpt_text(report_text)
    return to_documents(parsed_paths)

Writing rpt_parser.py


In [5]:
%%writefile vector_store.py
"""
vector_store.py

Manages vector store lifecycle using HuggingFace embeddings and FAISS.
Updated to use langchain-huggingface.
"""

import os
from typing import List

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Use the dedicated langchain-huggingface package
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from config import CONFIG


def get_embedding_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=CONFIG.EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_vector_store(docs: List[Document], save_path: str = CONFIG.FAISS_INDEX_PATH) -> FAISS:
    if not docs:
        raise ValueError("No documents provided to build vector store.")

    embeddings = get_embedding_model()
    vector_store = FAISS.from_documents(docs, embeddings)

    os.makedirs(save_path, exist_ok=True)
    vector_store.save_local(save_path)
    return vector_store


def load_vector_store(load_path: str = CONFIG.FAISS_INDEX_PATH) -> FAISS:
    if not os.path.exists(load_path):
        raise FileNotFoundError(f"No FAISS index found at '{load_path}'.")

    embeddings = get_embedding_model()
    return FAISS.load_local(
        load_path, embeddings, allow_dangerous_deserialization=True
    )


def get_or_create_vector_store(docs: List[Document] = None) -> FAISS:
    try:
        return load_vector_store()
    except FileNotFoundError:
        if docs is None:
            raise RuntimeError("No index on disk and no documents provided.")
        return build_vector_store(docs)

Writing vector_store.py


In [6]:
%%writefile llm_engine.py
"""
llm_engine.py

Handles LLM initialization and specialized prompt engineering for STA reports.
Updated to support ChatGroq.
"""

import os
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from config import CONFIG


def get_llm():
    """Returns a LangChain LLM instance based on config."""
    if CONFIG.LLM_PROVIDER == "groq":
        from langchain_groq import ChatGroq
        api_key = CONFIG.GROQ_API_KEY or os.getenv("GROQ_API_KEY")
        return ChatGroq(
            model_name=CONFIG.GROQ_MODEL,
            temperature=0.1,
            api_key=api_key,
        )
    elif CONFIG.LLM_PROVIDER == "ollama":
        from langchain_community.chat_models import ChatOllama
        return ChatOllama(
            model=CONFIG.OLLAMA_MODEL,
            base_url=CONFIG.OLLAMA_BASE_URL,
            temperature=0.1,
        )
    else:
        raise ValueError(f"Unsupported LLM provider: {CONFIG.LLM_PROVIDER}")


SYSTEM_PROMPT = """You are a senior ASIC physical design engineer with 15 years of experience in synthesis, static timing analysis (STA), and RTL optimization. You are analyzing timing path reports from Synopsys Design Compiler / PrimeTime.

Your task is to:
1. **Diagnose** the root cause of any timing violation based on the provided path data.
2. **Quantify** the problem using exact numbers from the report (slack values, cell delays, wire loads).
3. **Recommend** concrete, actionable fixes at the RTL or constraint level.

Rules:
- Always cite specific cell names, slack values, and delay contributions when making claims.
- If the path is clean (MET), briefly confirm and suggest margin optimization if relevant.
- If VIOLATED, categorize the violation: setup, hold, or max transition. Explain whether it is cell-delay dominated or interconnect-dominated.
- Suggest fixes in priority order: (1) RTL restructuring, (2) Pipeline insertion, (3) Cell sizing / Vt swap, (4) Constraint relaxation only as last resort.
- Use professional EDA terminology (WNS, TNS, fanout, transition time, clock skew, clock gating, etc.).
"""

HUMAN_PROMPT_TEMPLATE = """The user asked: {question}

Here are the retrieved timing paths from the synthesis report:

{context}

---

Provide a structured analysis with the following sections:
### 1. Path Summary
- Startpoint, Endpoint, Path Group, Module
- Slack status and value
- Dominant delay type (cell vs. net/interconnect)

### 2. Root-Cause Diagnosis
- Which cell(s) or net segment(s) dominate the delay?
- Is there excessive fanout, long wire load, or a slow cell type?
- Any clock skew or transition time issues?

### 3. Recommended Fixes
- RTL changes (e.g., pipeline registers, logic restructuring)
- Constraint changes (e.g., clock uncertainty, false paths, multicycle paths)
- Physical fixes (e.g., cell sizing, buffer insertion, placement bounds)

### 4. Confidence & Caveats
- State your confidence level and note any missing information that would help the diagnosis.
"""


def build_analysis_chain(retriever):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT_TEMPLATE),
    ])

    def format_docs(docs):
        formatted = []
        for i, doc in enumerate(docs, 1):
            meta = doc.metadata
            header = (
                f"[Path {i}] {meta.get('startpoint','?')} -> {meta.get('endpoint','?')} | "
                f"Slack: {meta.get('slack_value','?')} ({meta.get('slack_status','?')}) | "  # Fixed syntax here
                f"Module: {meta.get('module','?')}"
            )
            formatted.append(f"{header}\n{doc.page_content}\n")
        return "\n---\n".join(formatted)

    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | get_llm()
        | StrOutputParser()
    )
    return chain

Writing llm_engine.py


In [7]:
%%writefile retriever.py
"""
retriever.py

Configures retriever for EDA timing reports with backward/forward compatibility across LangChain versions.
"""

from langchain_community.vectorstores import FAISS
from config import CONFIG
from llm_engine import get_llm

# Handle LangChain package restructuring (v0.3+ / v1.0+)
HAS_SELF_QUERY = False
try:
    from langchain.chains.query_constructor.base import AttributeInfo
    from langchain.retrievers.self_query.base import SelfQueryRetriever
    HAS_SELF_QUERY = True
except ModuleNotFoundError:
    try:
        from langchain_classic.chains.query_constructor.base import AttributeInfo
        from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
        HAS_SELF_QUERY = True
    except ModuleNotFoundError:
        HAS_SELF_QUERY = False


if HAS_SELF_QUERY:
    METADATA_FIELD_INFO = [
        AttributeInfo(
            name="slack_status",
            description="Timing closure status. Must be explicitly 'VIOLATED' or 'MET'.",
            type="string",
        ),
        AttributeInfo(
            name="slack_value",
            description="Timing slack in nanoseconds (e.g. -0.42 or 0.15).",
            type="float",
        ),
        AttributeInfo(
            name="module",
            description="Top-level design or module name.",
            type="string",
        ),
        AttributeInfo(
            name="startpoint",
            description="Register or port where the timing path begins.",
            type="string",
        ),
        AttributeInfo(
            name="endpoint",
            description="Register or port where the timing path ends.",
            type="string",
        ),
        AttributeInfo(
            name="path_group",
            description="Clock domain or constraint group.",
            type="string",
        ),
        AttributeInfo(
            name="top_cell_1",
            description="Instance name of the cell with highest incremental delay.",
            type="string",
        ),
        AttributeInfo(
            name="top_cell_1_delay",
            description="Incremental delay contribution of top_cell_1 in nanoseconds.",
            type="float",
        ),
    ]

    DOCUMENT_CONTENT_DESCRIPTION = (
        "A single timing path extracted from an ASIC synthesis report (.rpt)."
    )


def get_self_query_retriever(vector_store: FAISS):
    """
    Returns a SelfQueryRetriever if supported, or falls back to standard FAISS vector retriever.
    """
    if HAS_SELF_QUERY:
        try:
            llm = get_llm()
            return SelfQueryRetriever.from_llm(
                llm=llm,
                vectorstore=vector_store,
                document_contents=DOCUMENT_CONTENT_DESCRIPTION,
                metadata_field_info=METADATA_FIELD_INFO,
                verbose=CONFIG.SELF_QUERY_VERBOSE,
                search_kwargs={"k": CONFIG.TOP_K_PATHS},
                enable_limit=False,
            )
        except Exception as e:
            print(f"[Warning] Could not initialize SelfQueryRetriever ({e}). Falling back to FAISS vector retriever.")

    # Fallback to standard similarity search retriever
    return vector_store.as_retriever(search_kwargs={"k": CONFIG.TOP_K_PATHS})

Writing retriever.py


In [8]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.2 MB/s eta 0:00:00:00:01


In [10]:
%%writefile config.py
"""
config.py
Configuration settings for EDA STA Analyzer.
"""

class Config:
    LLM_PROVIDER = "groq"
    # llama-3.1-8b-instant has a 30,000 TPM limit on Groq Free Tier (vs 12,000 on 70B)
    GROQ_MODEL = "llama-3.1-8b-instant"
    GROQ_API_KEY = None
    FAISS_INDEX_PATH = "./temp_faiss"
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    TOP_K_PATHS = 2  # Retrieve max 2 paths to keep prompt size tiny
    SELF_QUERY_VERBOSE = False

CONFIG = Config()

Overwriting config.py


In [11]:
%%writefile llm_engine.py
"""
llm_engine.py
Handles LLM initialization and specialized prompt engineering for STA reports.
"""

import os
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from config import CONFIG


def get_llm():
    """Returns a LangChain LLM instance explicitly locked to llama-3.1-8b-instant."""
    if CONFIG.LLM_PROVIDER == "groq":
        from langchain_groq import ChatGroq
        api_key = CONFIG.GROQ_API_KEY or os.getenv("GROQ_API_KEY")
        return ChatGroq(
            model_name="llama-3.1-8b-instant",  # Hardcoded to prevent fallback to 70B
            temperature=0.1,
            api_key=api_key,
        )
    else:
        raise ValueError(f"Unsupported LLM provider: {CONFIG.LLM_PROVIDER}")


SYSTEM_PROMPT = """You are a senior ASIC physical design engineer analyzing STA timing reports.
Diagnose root cause violations, quantify slack/delays, and suggest concise RTL/constraint/cell-sizing fixes."""

HUMAN_PROMPT_TEMPLATE = """User question: {question}

Retrieved timing paths:
{context}

Provide a concise analysis:
1. Path Summary (Startpoint, Endpoint, Slack)
2. Root-Cause Diagnosis (Dominant cells/nets)
3. Recommended Fixes
"""


def build_analysis_chain(retriever):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT_TEMPLATE),
    ])

    def format_docs(docs):
        formatted = []
        for i, doc in enumerate(docs, 1):
            meta = doc.metadata
            header = (
                f"[Path {i}] {meta.get('startpoint','?')} -> {meta.get('endpoint','?')} | "
                f"Slack: {meta.get('slack_value','?')} ({meta.get('slack_status','?')})"
            )
            # Strictly limit each path to 800 characters (~200 tokens)
            truncated_content = doc.page_content[:800]
            formatted.append(f"{header}\n{truncated_content}\n")
        return "\n---\n".join(formatted)

    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | get_llm()
        | StrOutputParser()
    )
    return chain

Overwriting llm_engine.py


In [ ]:
import os
import threading
from fastapi import FastAPI, File, Form, UploadFile
import nest_asyncio
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

# Import your existing module logic
from llm_engine import build_analysis_chain
from retriever import get_self_query_retriever
from rpt_parser import parse_rpt_text, to_documents
from vector_store import build_vector_store

app = FastAPI(title="EDA STA Analyzer Backend")

# In-memory store for active session
server_state = {"chain": None, "retriever": None, "docs": []}


class QueryRequest(BaseModel):
  question: str


@app.post("/upload")
async def upload_file(
    file: UploadFile = File(...), groq_api_key: str = Form(...)
):
  os.environ["GROQ_API_KEY"] = groq_api_key

  # 1. Read file contents sent from Local Streamlit
  content = (await file.read()).decode("utf-8", errors="ignore")

  # 2. Run existing parser and build vector store
  parsed_paths = parse_rpt_text(content)
  docs = to_documents(parsed_paths)
  vector_store = build_vector_store(docs, save_path="./temp_faiss")
  retriever = get_self_query_retriever(vector_store)
  chain = build_analysis_chain(retriever)

  # Save to state
  server_state["chain"] = chain
  server_state["retriever"] = retriever
  server_state["docs"] = docs

  # Compute stats to send back to local UI
  df_meta = [d.metadata for d in docs]
  total = len(docs)
  violated = sum(1 for m in df_meta if m.get("slack_status") == "VIOLATED")
  slack_vals = [
      m.get("slack_value")
      for m in df_meta
      if m.get("slack_value") is not None
  ]
  wns = min(slack_vals) if slack_vals else 0.0

  return {
      "status": "success",
      "total_paths": total,
      "violations": violated,
      "wns": wns,
  }


@app.post("/query")
async def query_endpoint(req: QueryRequest):
  chain = server_state.get("chain")
  if not chain:
    return {
        "status": "error",
        "message": "No report uploaded yet on Kaggle server.",
    }

  # Execute LangChain pipeline
  response = chain.invoke(req.question)
  return {"status": "success", "answer": response}


# --- Start Server & Ngrok ---
nest_asyncio.apply()

NGROK_TOKEN = "paste_ngrok_authentication_token_here"  
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

# Open Tunnel on port 8000
public_url = ngrok.connect(8000)
print("=" * 70)
print(f"KAGGLE BACKEND URL: {public_url}")
print("Copy this URL and paste it into your local Streamlit app sidebar!")
print("=" * 70)


def run_fastapi():
  uvicorn.run(app, host="0.0.0.0", port=8000)


threading.Thread(target=run_fastapi, daemon=True).start()

KAGGLE BACKEND URL: NgrokTunnel: "https://utilize-seventeen-uplifting.ngrok-free.dev" -> "http://localhost:8000"
Copy this URL and paste it into your local Streamlit app sidebar!


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /query HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     41.37.248.244:0 - "POST /upload HTTP/1.1" 200 OK
INFO: 